# Flows

Declare transition flows over a stratified map, compile once to a
`CompiledModel`, query edges with `Source` / `Dest`, and step a JAX vector
field. `euler` returns the final state only.

In [ ]:
import numpy as np

from summer4 import (
    Dest,
    FlowModel,
    Overwrite,
    Property,
    PropertyMap,
    Source,
    TraitChain,
    TransitionFlow,
    euler,
)

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))
severity = Property("severity", ("mild", "severe"))

pmap = (
    PropertyMap.from_property(state)
    .stratify(age)
    .stratify(severity, where=state["I"])
)
assert pmap.size == 12
model = FlowModel(pmap)

## Infection with a severity split and an age overwrite

Infection fans out across severity (`split=`). An `Overwrite` with `where=`
zeros infection in the youngest band. Recovery matches leftover age.

In [ ]:
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        0.2,
        split={severity: {"mild": 0.7, "severe": 0.3}},
        adjust=[Overwrite(0.0, where=age["0-4"])],
    )
)
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.1))

## Ageing as one named flow

`TraitChain` lists every band step on one flow. Leftover properties still match.

In [ ]:
model.add_flow(
    TransitionFlow(
        "ageing",
        age.present(),
        age.present(),
        0.2,
        pairing=TraitChain(age, (("0-4", "5-9"), ("5-9", "10+"))),
    )
)

## Compile and query edges with Source / Dest

In [ ]:
compiled = model.compile()
infection = compiled.edges("infection")

assert infection.n_edges == 6
labels = infection.labels()
assert "state=S_age=5-9 -> state=I_age=5-9_severity=mild" in labels

young = infection.select(Source(age["0-4"]))
assert young.size == 2
mild_from_mid = infection.select(Source(age["5-9"]) & Dest(severity["mild"]))
assert mild_from_mid.size == 1

assert infection.moves_mask(state).all()
assert not infection.moves_mask(age).any()

try:
    infection.select(state["S"])
except TypeError as exc:
    assert "Source" in str(exc)

## Step the vector field

Young susceptibles do not infect (the overwrite); ageing still moves them.
Transition-only models conserve total mass.

In [ ]:
y0 = np.zeros(pmap.size)
y0[pmap.select(state["S"])] = 100.0
y0[pmap.select(state["I"] & severity["mild"])] = 10.0

dy = np.asarray(compiled.vector_field(0.0, y0, {}))
assert np.isclose(dy.sum(), 0.0)
s_young = pmap.select_one(state["S"] & age["0-4"])
assert np.isclose(dy[s_young], -0.2 * y0[s_young])

y1 = np.asarray(euler(compiled.vector_field, 0.0, y0, {}, dt=0.5, steps=4))
assert np.isclose(y1.sum(), y0.sum())
assert (y1[pmap.select(state["R"])] > 0).any()